In [8]:
%%writefile app.py

import streamlit as st
import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage import measure
import base64, io, json, sys, os, shutil, traceback, tempfile
from google import genai
from google.genai import types
from ultralytics import YOLO, SAM
from ultralytics.models.sam import SAM3SemanticPredictor
from pydantic import BaseModel
from typing import Optional
from enum import StrEnum

st.set_page_config(page_title="Car Damage Detection", page_icon="🚗",
                   layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
    .top-navbar {
        position: fixed; top: 0; left: 0; right: 0; height: 52px;
        background: #161b22; border-bottom: 1px solid #30363d;
        display: flex; align-items: center; justify-content: space-between;
        padding: 0 24px; z-index: 999999;
    }
    .top-navbar .brand { font-weight: 700; font-size: 15px; color: #e6edf3; display: flex; align-items: center; gap: 8px; }
    .top-navbar .nav-links { display: flex; gap: 20px; align-items: center; }
    .top-navbar .nav-link { font-size: 13px; color: #8b949e; cursor: pointer; }
    .top-navbar .nav-link.active { color: #3fb950; border-bottom: 2px solid #3fb950; padding-bottom: 2px; }
    .top-navbar .nav-user { display: flex; align-items: center; gap: 10px; }
    .top-navbar .nav-avatar { width: 28px; height: 28px; border-radius: 50%; background: #238636;
        display: flex; align-items: center; justify-content: center; font-size: 11px; font-weight: 700; color: white; }
    .stApp { background-color: #0d1117; margin-top: 52px; }
    .stApp > header { top: 52px !important; }
    section[data-testid="stSidebar"] {
        background-color: #0d1117; border-right: 1px solid #30363d;
        min-width: 290px !important; transform: none !important; visibility: visible !important;
    }
    button[data-testid="collapsedControl"] { display: none !important; }
    html, body, [class*="css"] { font-family: 'Inter', sans-serif; color: #e6edf3; font-size: 13px; }
    h1 { font-size: 1.2rem !important; font-weight: 600 !important; color: #e6edf3 !important; margin-bottom: 0 !important; }
    h3 { font-size: 0.7rem !important; font-weight: 500 !important; color: #8b949e !important; text-transform: uppercase; letter-spacing: 0.08em; }
    /* botones — ancho completo */
    div[data-testid="stButton"],
    div[data-testid="stButton"] > button,
    div[data-testid="stButton"] > button > div,
    div[data-testid="stButton"] > button > div > p {
        width: 100% !important;
        text-align: center !important;
    }
    div[data-testid="stButton"] > button {
        border-radius: 6px !important;
        padding: 10px 0 !important;
        font-weight: 600 !important;
        font-size: 14px !important;
        margin-top: 4px !important;
        display: flex !important;
        justify-content: center !important;
        align-items: center !important;
        background-color: #238636 !important;
        color: white !important;
        border: none !important;
    }
    div[data-testid="stButton"] > button:hover {
        background-color: #2ea043 !important;
    }
    [data-testid="stMetric"] { background: #161b22; border: 1px solid #30363d; border-radius: 6px; padding: 0.5rem 0.75rem !important; }
    [data-testid="stMetricValue"] { color: #e6edf3 !important; font-size: 1.1rem !important; }
    [data-testid="stMetricLabel"] { color: #8b949e !important; font-size: 0.7rem !important; }
    [data-testid="stFileUploader"] { background: #161b22; border: 1px dashed #30363d; border-radius: 6px; }
    .block-container { padding-top: 0.5rem !important; padding-bottom: 1rem !important; padding-left: 2rem !important; padding-right: 2rem !important; }
    div[data-testid="stVerticalBlock"] > div { gap: 0.4rem !important; }
    hr { margin: 0.5rem 0 !important; }
    #MainMenu, footer, header { visibility: hidden; }
    .stSlider [data-testid="stTickBar"] { display: none; }
    .stSlider > div { padding: 0 !important; }
    [data-testid="stCheckbox"] label { color: #8b949e !important; font-size: 13px; }
    .log-box { background: #0d1117; border: 1px solid #30363d; border-radius: 6px;
        padding: 8px 12px; font-family: 'Courier New', monospace; font-size: 11px;
        color: #8b949e; max-height: 110px; overflow-y: auto; white-space: pre-wrap; line-height: 1.6; }
    .log-error { color: #f85149; } .log-ok { color: #3fb950; } .log-warn { color: #e3b341; }
    .preview-label { font-size: 11px; color: #8b949e; text-transform: uppercase; letter-spacing: 0.06em; margin-bottom: 4px; }
    [data-testid="stImage"] img { max-height: 480px; object-fit: contain; width: 100%; }
    [data-testid="stVideo"] { display: flex; justify-content: center; }
    [data-testid="stVideo"] video { max-height: 420px !important; width: 75% !important; border-radius: 8px; }
    div[data-testid="stRadio"] > div { flex-direction: row !important; gap: 12px !important; }
    div[data-testid="stRadio"] label { font-size: 13px !important; }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="top-navbar">
    <div class="nav-links">
        <div class="brand">🚗 InspectAI</div>
        <span class="nav-link">Inicio</span>
        <span class="nav-link active">Inspección</span>
        <span class="nav-link">Historial</span>
        <span class="nav-link">Reportes</span>
    </div>
    <div class="nav-user">
        <span style="font-size:12px;color:#8b949e;">MIOTI Aseguradora FK</span>
        <div class="nav-avatar">MH</div>
        <span style="font-size:13px;color:#e6edf3;">Mynor Hernández</span>
    </div>
</div>
""", unsafe_allow_html=True)

YOLO_PATH = '/content/drive/MyDrive/yolo26_weights/best.pt'
SAM_PATH  = '/content/drive/MyDrive/sam3_weights/sam3.pt'
COST_PATH = '/content/drive/MyDrive/cost_estimator'

@st.cache_resource(show_spinner=False)
def cargar_yolo(): return YOLO(YOLO_PATH)
@st.cache_resource(show_spinner=False)
def cargar_sam(): return SAM(SAM_PATH)
@st.cache_resource(show_spinner=False)
def cargar_sam_semantic():
    return SAM3SemanticPredictor(overrides=dict(model=SAM_PATH, conf=0.5, iou=0.5, imgsz=1024))
@st.cache_resource(show_spinner=False)
def cargar_estimator():
    if not os.path.exists('/content/cost_estimator'):
        shutil.copytree(COST_PATH, '/content/cost_estimator', dirs_exist_ok=True)
    if '/content' not in sys.path: sys.path.insert(0, '/content')
    from cost_estimator.cost_estimator import CostEstimator
    return CostEstimator(
        repair_costs_path='/content/cost_estimator/data/repair_costs.json',
        prices_csv_path='/content/cost_estimator/data/prices_dataset.csv',
        severidad_csv_path='/content/cost_estimator/data/severidad.csv',
    )
@st.cache_resource(show_spinner=False)
def cargar_gemini():
    return genai.Client(api_key=os.environ.get('GEMINI_KEY', ''))

class ClaseDano(StrEnum):
    SCRATCH      = "scratch"
    DENT         = "dent"
    BROKEN_GLASS = "glass shatter"
    CRACK        = "crack"
    LAMP_BROKEN  = "lamp broken"
    TIRE_FLAT    = "tire flat"
class ZonaAuto(StrEnum):
    DOOR="door"; HOOD="hood"; BUMPER="bumper"; FENDER="fender"
    WINDSHIELD="windshield"; ROOF="roof"; TRUNK="trunk"
    WHEEL="wheel"; HEADLIGHT="headlight"; MIRROR="mirror"
class IntencionParseada(BaseModel):
    clases: list[ClaseDano]
    zonas: list[ZonaAuto] = []

ZONA_A_PROMPT = {
    "door":"car door","hood":"car hood","bumper":"car bumper","fender":"car fender",
    "windshield":"car windshield","roof":"car roof","trunk":"car trunk",
    "wheel":"car wheel","headlight":"car headlight","mirror":"car side mirror",
}
ZONA_NOMBRE_ES = {
    "door":"puerta","hood":"capó","bumper":"parachoques","fender":"guardabarros",
    "windshield":"parabrisas","roof":"techo","trunk":"maletero",
    "wheel":"rueda","headlight":"faro","mirror":"espejo",
}
ZONA_EMOJI = {
    "door":"🚪","hood":"🔧","bumper":"⬛","fender":"🔩",
    "windshield":"🪟","roof":"🏠","trunk":"📦","wheel":"🛞","headlight":"💡","mirror":"🔍",
}
COLORES_CLASE = {
    "scratch":      (55,138,221),
    "dent":         (226,75,74),
    "crack":        (227,179,65),
    "glass shatter":(63,185,80),
    "lamp broken":  (210,120,50),
    "tire flat":    (180,80,180),
}
UMBRAL_BAJO=0.1; UMBRAL_MEDIO=0.5
SEV_COLOR={"alto":"🔴","medio":"🟡","bajo":"🟢","n/a":"⚪"}

for k,v in [("logs",[]),("img_dets",None),("img_modelo",""),("img_costos",None),
            ("img_rechazadas",[]),("imagen_bytes",None),("imagen_nombre",""),
            ("img_renders",None),("vid_resultados",None),
            ("vid_resumen",None),("vid_output_path",None)]:
    if k not in st.session_state: st.session_state[k] = v

def log(msg, tipo="info"):
    prefix = {"info":"→","ok":"✓","error":"✗","warn":"⚠"}.get(tipo,"→")
    st.session_state["logs"].append((tipo, f"{prefix} {msg}"))
def render_logs():
    if not st.session_state["logs"]: return "<div class='log-box'>Sin logs.</div>"
    lineas = []
    for tipo, msg in st.session_state["logs"]:
        if tipo=="error": lineas.append(f"<span class='log-error'>{msg}</span>")
        elif tipo=="ok":  lineas.append(f"<span class='log-ok'>{msg}</span>")
        elif tipo=="warn":lineas.append(f"<span class='log-warn'>{msg}</span>")
        else:             lineas.append(msg)
    return "<div class='log-box'>" + "<br>".join(lineas) + "</div>"

def preparar_imagen(src):
    img = Image.open(src).convert('RGB')
    return np.array(ImageOps.exif_transpose(img))
def calcular_severidad(mascara, imagen):
    pct = float(mascara.sum())/(imagen.shape[0]*imagen.shape[1])*100
    if pct < UMBRAL_BAJO: return "bajo", pct
    elif pct < UMBRAL_MEDIO: return "medio", pct
    else: return "alto", pct
def calcular_severidad_bbox(caja, imagen):
    h,w = imagen.shape[:2]; x1,y1,x2,y2 = caja.astype(int)
    pct = (x2-x1)*(y2-y1)/(h*w)*100
    if pct < UMBRAL_BAJO: return "bajo", pct
    elif pct < UMBRAL_MEDIO: return "medio", pct
    else: return "alto", pct
def recortar_region(imagen, caja, padding=30):
    h,w = imagen.shape[:2]; x1,y1,x2,y2 = caja.astype(int)
    return imagen[max(0,y1-padding):min(h,y2+padding), max(0,x1-padding):min(w,x2+padding)]
def imagen_a_b64(arr):
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()
def iou(a, b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    inter = max(0,min(ax2,bx2)-max(ax1,bx1))*max(0,min(ay2,by2)-max(ay1,by1))
    if inter==0: return 0.0
    return float(float(inter)/float((ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter))
def deduplicar(dets, umbral=0.5):
    resultado = []
    for det in dets:
        if not any(det["clase"]==r["clase"] and iou(det["caja"].tolist(),r["caja"].tolist())>umbral for r in resultado):
            resultado.append(det)
    return resultado

def parse_intent(prompt, client):
    instrucciones = """Parser para detección de daños en autos.
Si menciona múltiples zonas, inclúyelas TODAS en zonas. Sin zona → zonas=[]."""
    try:
        r = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=f"{instrucciones}\n\nPetición: \"{prompt}\"",
            config=types.GenerateContentConfig(temperature=0,
                response_mime_type="application/json", response_schema=IntencionParseada))
        return IntencionParseada.model_validate_json(r.text)
    except Exception as e:
        log(f"Error intent: {e}", "error")
        return IntencionParseada(clases=list(ClaseDano), zonas=[])

def detectar_zona(imagen, zona, sam_sem):
    p = ZONA_A_PROMPT.get(str(zona))
    if not p: return None
    sam_sem.prompts = {"text": [p]}
    results = sam_sem(source=imagen)
    if results[0].boxes is None or len(results[0].boxes)==0: return None
    cajas = results[0].boxes.xyxy.cpu().numpy().astype(int)
    confs = results[0].boxes.conf.cpu().numpy()
    idx = confs.argmax()
    return cajas[idx].tolist() if float(confs[idx]) >= 0.7 else None

def recortar_zona(imagen, caja, padding=20):
    h,w=imagen.shape[:2]; x1,y1,x2,y2=caja
    return imagen[max(0,y1-padding):min(h,y2+padding),
                  max(0,x1-padding):min(w,x2+padding)], (max(0,x1-padding),max(0,y1-padding))

def detectar_yolo(imagen, conf, yolo_model):
    results = yolo_model.predict(source=imagen, conf=conf, verbose=False)
    nombres = yolo_model.names
    return [{"clase":nombres[int(cls)],"caja":box.cpu().numpy(),"confianza":float(c)}
            for box,cls,c in zip(results[0].boxes.xyxy,results[0].boxes.cls,results[0].boxes.conf)]

def segmentar(imagen, dets, sam_model):
    if not dets: return []
    try:
        res = sam_model.predict(source=imagen, bboxes=[d["caja"].tolist() for d in dets], verbose=False)
        if res[0].masks is not None:
            n=len(res[0].masks.data)
            mascaras=[res[0].masks.data[i].cpu().numpy() if i<n else None for i in range(len(dets))]
            log(f"SAM3 batch → {n}/{len(dets)} máscaras", "ok")
        else:
            mascaras=[None]*len(dets); log("SAM3 → sin máscaras", "warn")
    except Exception as e:
        mascaras=[None]*len(dets); log(f"Error SAM3: {e}", "error")
    return [{**det,"mascara":m} for det,m in zip(dets,mascaras)]

def juez_vlm(imagen, det, client):
    buf=io.BytesIO(); Image.fromarray(recortar_region(imagen,det["caja"])).save(buf,format='JPEG',quality=90)
    b64=base64.b64encode(buf.getvalue()).decode(); clase=det["clase"]
    prompt=f"""Experto en peritaje vehicular. ¿Es este {clase} daño real o falso positivo?
JSON: {{"es_dano":true/false,"confianza":0.0-1.0,"razon":"breve"}}"""
    try:
        resp=client.models.generate_content(model="gemini-3.1-flash-lite",
            contents=[{"parts":[{"inline_data":{"mime_type":"image/jpeg","data":b64}},{"text":prompt}]}])
        raw=resp.text.strip()
        if raw.startswith("```"): raw=raw.split("```")[1]; raw=raw[4:] if raw.lower().startswith("json") else raw
        return json.loads(raw.strip())
    except Exception as e:
        err_str = str(e)
        if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
            import time; time.sleep(5)
            try:
                resp=client.models.generate_content(model="gemini-3.1-flash-lite",
                    contents=[{"parts":[{"inline_data":{"mime_type":"image/jpeg","data":b64}},{"text":prompt}]}])
                raw=resp.text.strip()
                if raw.startswith("```"): raw=raw.split("```")[1]; raw=raw[4:] if raw.lower().startswith("json") else raw
                return json.loads(raw.strip())
            except:
                pass
        log(f"Error juez ({clase}): {e}","error")
        return {"es_dano":True,"confianza":0.5,"razon":"error — conservada"}

def identificar_modelo(imagen, client):
    prompt = """Eres un experto en identificación de vehículos.
Analiza la imagen e identifica la marca, modelo y año aproximado del auto.
Responde SOLO con el formato: 'Marca Modelo Año' (ej: 'Toyota Corolla 2018').
Si no puedes identificar la marca con certeza, responde exactamente: 'Genérico'
No añadas explicaciones ni texto adicional."""
    try:
        resp=client.models.generate_content(model="gemini-3.1-flash-lite",
            contents=[{"parts":[{"inline_data":{"mime_type":"image/jpeg","data":imagen_a_b64(imagen)}},
                                {"text":prompt}]}])
        resultado = resp.text.strip()
        # si la respuesta tiene más de 6 palabras probablemente es una explicación
        if len(resultado.split()) > 6 or "no" in resultado.lower()[:10]:
            return "Genérico"
        return resultado
    except Exception as e:
        log(f"Error modelo: {e}","error"); return "Genérico"

def estimar_costos(dets, imagen, modelo_auto, estimator):
    partes=modelo_auto.strip().split()
    dets_fmt=[{"class_name":d["clase"],"bbox":d["caja"].tolist(),"confidence":d["confianza"]} for d in dets]
    if not dets_fmt: return None
    try:
        return estimator.estimate(detections=dets_fmt, image_shape=imagen.shape,
            make=partes[0] if partes else None,
            model=partes[1] if len(partes)>1 else None,
            year=partes[2] if len(partes)>2 else None)
    except Exception as e:
        log(f"Error costos: {e}","error"); return None

def renderizar_detecciones(imagen, dets, modo="Ambas"):
    fig,ax=plt.subplots(1,1,figsize=(14,9))
    ax.imshow(imagen); ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')
    mostrar_bb=modo in("Bounding boxes","Ambas"); mostrar_sam=modo in("Máscaras SAM","Ambas")
    for det in dets:
        clase=det["clase"]; caja=det["caja"]; mascara=det.get("mascara")
        color_01=tuple(c/255 for c in COLORES_CLASE.get(clase,(140,140,140)))
        if mascara is not None and mostrar_sam:
            overlay=np.zeros((*mascara.shape,4)); overlay[mascara>0]=[*color_01,0.40]
            ax.imshow(overlay,extent=[0,imagen.shape[1],imagen.shape[0],0])
            for c in measure.find_contours(mascara,0.5): ax.plot(c[:,1],c[:,0],color=color_01,linewidth=1.8)
        sev,pct=calcular_severidad(mascara,imagen) if mascara is not None else calcular_severidad_bbox(caja,imagen)
        if mostrar_bb:
            x1,y1,x2,y2=caja.astype(int)
            ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor=color_01,
                         facecolor='none',linestyle='--' if (mascara is not None and mostrar_sam) else '-'))
            ax.text(x1,y1-5,f"{clase} | {sev} | {det['confianza']:.2f}",color=color_01,fontsize=9,
                    fontweight='bold',bbox=dict(facecolor='#0d1117',alpha=0.75,pad=2,edgecolor='none'))
        elif mostrar_sam and mascara is not None:
            contornos=measure.find_contours(mascara,0.5)
            if contornos:
                y0,x0=contornos[0][0]
                ax.text(x0,y0-5,f"{clase} | {sev} | {det['confianza']:.2f}",color=color_01,fontsize=9,
                        fontweight='bold',bbox=dict(facecolor='#0d1117',alpha=0.75,pad=2,edgecolor='none'))
    ax.axis('off'); plt.tight_layout(pad=0)
    buf=io.BytesIO(); fig.savefig(buf,format='png',dpi=130,bbox_inches='tight',facecolor='#0d1117',edgecolor='none')
    plt.close(fig); buf.seek(0)
    return np.array(Image.open(buf))

def mostrar_desglose(dets, imagen, costos):
    for i,det in enumerate(dets):
        mascara=det.get("mascara")
        sev,pct=calcular_severidad(mascara,imagen) if mascara is not None else calcular_severidad_bbox(det["caja"],imagen)
        recorte=recortar_region(imagen,det["caja"]); juez=det.get("juez",{})
        c_thumb,c_info,c_sev,c_conf,c_costo=st.columns([1,3,1.5,1.5,2])
        with c_thumb: st.image(recorte,width=56)
        with c_info:
            st.markdown(f"**{det['clase'].capitalize()}**")
            if costos and i<len(costos.get("items",[])): st.caption(costos["items"][i].get("part_name",""))
            if juez.get("razon"):
                with st.expander("Razón del juez",expanded=False): st.caption(f"_{juez['razon']}_")
        with c_sev:
            st.markdown(f"{SEV_COLOR.get(sev,'⚪')} **{sev}**{'*' if mascara is None else ''}")
            st.caption(f"{pct:.2f}% área")
        with c_conf: st.progress(det["confianza"]); st.caption(f"{det['confianza']:.2f}")
        with c_costo:
            if costos and i<len(costos.get("items",[])):
                item=costos["items"][i]; st.markdown(f"**{item['cost_min']}–{item['cost_max']}** {item['currency']}")
            else: st.markdown("—")
        st.divider()

# ── sidebar dinámico ──────────────────────────────────────────────────────────
with st.sidebar:
    # tabs visuales como botones
    if "modo_selector" not in st.session_state:
        st.session_state["modo_selector"] = "🖼️  Imagen"
    col_t1, col_t2 = st.columns(2)
    with col_t1:
        if st.button("🖼️  Imagen", key="btn_modo_img",
                     type="primary" if st.session_state["modo_selector"]=="🖼️  Imagen" else "secondary",
                     use_container_width=True):
            st.session_state["modo_selector"] = "🖼️  Imagen"
    with col_t2:
        if st.button("🎥  Video", key="btn_modo_vid",
                     type="primary" if st.session_state["modo_selector"]=="🎥  Video" else "secondary",
                     use_container_width=True):
            st.session_state["modo_selector"] = "🎥  Video"
    modo = st.session_state["modo_selector"]
    st.markdown("---")

    if modo == "🖼️  Imagen":
        st.markdown("### Inspección de vehículo")
        upload = st.file_uploader("Imagen", type=["jpg","jpeg","png","webp"], key="img_upload")
        if upload is not None:
            img_bytes = upload.read()
            st.session_state["imagen_bytes"] = img_bytes
            st.session_state["imagen_nombre"] = upload.name
            st.session_state["img_dets"] = None
            st.session_state["img_renders"] = None
        if st.session_state["imagen_bytes"] is not None:
            st.markdown("<p class='preview-label'>Vista previa</p>", unsafe_allow_html=True)
            st.image(io.BytesIO(st.session_state["imagen_bytes"]), use_container_width=True)
        prompt = st.text_area("Prompt", placeholder="Revisa los daños en el capó, puerta y parabrisas...",
                               height=85, key="img_prompt")
        st.markdown("### Opciones")
        usar_juez   = st.checkbox("Juez VLM (filtro FP)", value=True)
        usar_zona   = st.checkbox("Detección de zona", value=True)
        usar_costos = st.checkbox("Estimación de costos", value=True)
        conf_umbral = st.slider("Confianza YOLO", 0.05, 0.50, 0.20, 0.05, format="%.2f")
        analizar = st.button("🔍  Analizar daños")
        if analizar:
            st.session_state["img_dets"] = None
            st.session_state["img_renders"] = None
    else:
        st.markdown("### Inspección por video")
        video_upload = st.file_uploader("Video del vehículo", type=["mp4","mov","avi"], key="vid_upload")
        if video_upload is not None:
            vid_bytes_up = video_upload.read()
            st.session_state["vid_file_bytes"] = vid_bytes_up
            st.session_state["vid_file_name"]  = video_upload.name
            vid_tmp_path = "/content/uploaded_video.mp4"
            with open(vid_tmp_path, "wb") as _f: _f.write(vid_bytes_up)
            st.session_state["vid_tmp_path"] = vid_tmp_path
        st.markdown("### Opciones")
        conf_vid     = st.slider("Confianza YOLO", 0.05, 0.50, 0.25, 0.05, format="%.2f", key="vid_conf")
        analizar_vid = st.button("🎥  Analizar video", key="btn_vid")
        if analizar_vid:
            st.session_state["vid_resultados"]  = None
            st.session_state["vid_resumen"]     = None
            st.session_state["vid_output_path"] = None
            st.session_state["logs"]            = []
        # placeholders para que imagen no falle fuera del if
        analizar = False

    st.markdown("---")
    st.markdown("### Acerca del sistema")
    st.caption("YOLO26l fine-tuned · SAM 3 · Gemini gemini-3.1-flash-lite · Cost Estimator")
    st.caption("Proyecto Final — MIOTI Master IA Avanzada")

# ── área principal ────────────────────────────────────────────────────────────
st.markdown("# Car Damage Detection")
st.markdown("YOLO26 · SAM 3 · Gemini · Cost Estimator")
st.markdown("---")

# ══════════════════════════════════════════════════════════════════════════════
# MODO IMAGEN
# ══════════════════════════════════════════════════════════════════════════════
if modo == "🖼️  Imagen":
    if st.session_state["imagen_bytes"] is None:
        st.info("Sube una imagen del vehículo en el panel izquierdo para comenzar.")
        st.stop()

    imagen = preparar_imagen(io.BytesIO(st.session_state["imagen_bytes"]))
    imagen_ph = st.empty()

    if st.session_state["img_renders"] is not None:
        imagen_ph.image(st.session_state["img_renders"],
                        caption="Detecciones — YOLO26 + SAM 3", use_container_width=True)
    else:
        imagen_ph.image(imagen, caption=st.session_state.get("imagen_nombre",""),
                        use_container_width=True)

    if analizar:
        st.session_state["logs"] = []
        st.markdown("### Cargando modelos")
        bar=st.progress(0); estado_txt=st.empty()
        estado_txt.markdown("Cargando YOLO26..."); bar.progress(15)
        yolo_model=cargar_yolo(); log("YOLO26 listo","ok")
        estado_txt.markdown("Cargando SAM 3..."); bar.progress(40)
        sam_model=cargar_sam(); log("SAM 3 listo","ok")
        estado_txt.markdown("Cargando detector de zonas..."); bar.progress(60)
        sam_sem=cargar_sam_semantic(); log("SAM 3 semantic listo","ok")
        estado_txt.markdown("Conectando Gemini..."); bar.progress(78)
        client=cargar_gemini(); log("Gemini conectado","ok")
        estado_txt.markdown("Cargando base de costos..."); bar.progress(92)
        estimator=cargar_estimator() if usar_costos else None
        if estimator: log("CostEstimator listo","ok")
        bar.progress(100); estado_txt.markdown("✅ Modelos listos")

        st.markdown("---")
        col_prog,col_logs=st.columns([3,2])
        with col_prog:
            st.markdown("### Analizando el vehículo")
            st.caption("Este proceso puede tardar hasta 2 minutos dependiendo del número de daños detectados.")
            pipeline_bar=st.progress(0); pipeline_txt=st.empty(); pipeline_det=st.empty()
        with col_logs:
            st.markdown("### Logs técnicos")
            logs_ph=st.empty(); logs_ph.markdown(render_logs(),unsafe_allow_html=True)

        prompt_texto = prompt.strip() or "revisa todos los daños del auto"

        try:
            pipeline_txt.markdown("🔍 **Interpretando tu petición...**"); pipeline_bar.progress(10)
            intencion=parse_intent(prompt_texto,client)
            clases_str=[str(c) for c in intencion.clases]; zonas=intencion.zonas
            if not clases_str:
                clases_str=[str(c) for c in ClaseDano]
                log("clases vacías — usando todas por defecto","warn")
            log(f"parse_intent → clases={clases_str}","ok")
            log(f"parse_intent → zonas={[str(z) for z in zonas]}","ok")
            logs_ph.markdown(render_logs(),unsafe_allow_html=True)

            todas_dets=[]
            if usar_zona and zonas:
                zonas_resumen=" · ".join(f"{ZONA_EMOJI.get(str(z),'📍')} {ZONA_NOMBRE_ES.get(str(z),str(z))}" for z in zonas)
                pipeline_det.markdown(f"Zonas a inspeccionar: **{zonas_resumen}**")
                paso_base=15; paso_zona=int(50/max(len(zonas),1))
                for idx_zona,zona in enumerate(zonas):
                    zona_str=str(zona); zona_es=ZONA_NOMBRE_ES.get(zona_str,zona_str); emoji=ZONA_EMOJI.get(zona_str,"📍")
                    pipeline_txt.markdown(f"{emoji} **Inspeccionando {zona_es}...**")
                    pipeline_bar.progress(paso_base+idx_zona*paso_zona)
                    pipeline_det.markdown(f"Localizando {zona_es} en la imagen...")
                    log(f"SAM3 semantic → buscando '{zona_str}'","info"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                    caja_zona=detectar_zona(imagen,zona,sam_sem)
                    if caja_zona:
                        imagen_crop,offset=recortar_zona(imagen,caja_zona)
                        log(f"Zona '{zona_str}' localizada: bbox={caja_zona}","ok")
                        log(f"YOLO26 → analizando '{zona_str}', conf={conf_umbral}","info")
                        logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                        dets_zona=detectar_yolo(imagen_crop,conf_umbral,yolo_model)
                        ox,oy=offset
                        for d in dets_zona: d["caja"][0]+=ox;d["caja"][2]+=ox;d["caja"][1]+=oy;d["caja"][3]+=oy
                        dets_zona=[d for d in dets_zona if d["clase"] in clases_str]
                        if dets_zona:
                            resumen=", ".join(f"{d['clase']} ({d['confianza']:.2f})" for d in dets_zona)
                            pipeline_det.markdown(f"**{emoji} {zona_es.capitalize()}** — {len(dets_zona)} daño(s): {resumen}")
                            log(f"YOLO '{zona_str}' → {[d['clase'] for d in dets_zona]}","ok")
                        else:
                            pipeline_det.markdown(f"**{emoji} {zona_es.capitalize()}** — sin daños")
                            log(f"YOLO '{zona_str}' → sin detecciones","warn")
                        todas_dets.extend(dets_zona)
                    else:
                        pipeline_det.markdown(f"**{emoji} {zona_es.capitalize()}** — zona no localizada")
                        log(f"Zona '{zona_str}' no detectada (conf < 0.7)","warn")
                    logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                antes=len(todas_dets); todas_dets=deduplicar(todas_dets)
                if antes!=len(todas_dets):
                    log(f"Deduplicación → {antes}→{len(todas_dets)}","ok")
                    logs_ph.markdown(render_logs(),unsafe_allow_html=True)
            else:
                pipeline_txt.markdown("🔎 **Detectando daños...**"); pipeline_bar.progress(30)
                log(f"YOLO26 → imagen completa, conf={conf_umbral}","info"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                todas_dets=detectar_yolo(imagen,conf_umbral,yolo_model)
                todas_dets=[d for d in todas_dets if d["clase"] in clases_str]
                log(f"YOLO26 → {len(todas_dets)} detecciones","ok")
                for d in todas_dets: log(f"  {d['clase']} conf={d['confianza']:.2f}","info")
                logs_ph.markdown(render_logs(),unsafe_allow_html=True)

            dets=todas_dets
            pipeline_txt.markdown("✂️ **Delimitando área de cada daño...**"); pipeline_bar.progress(65)
            log(f"SAM3 batch → {len(dets)} detecciones","info"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
            dets=segmentar(imagen,dets,sam_model); logs_ph.markdown(render_logs(),unsafe_allow_html=True)

            rechazadas=[]
            if usar_juez and dets:
                pipeline_txt.markdown("🧠 **Verificando daños reales...**"); pipeline_bar.progress(78)
                log(f"Juez VLM → evaluando {len(dets)} detecciones","info"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                validadas=[]
                for d in dets:
                    v=juez_vlm(imagen,d,client); d["juez"]=v
                    if v.get("es_dano") or v.get("confianza",1)<0.6:
                        validadas.append(d); log(f"  ✓ {d['clase']} (conf_juez={v.get('confianza',0):.2f})","ok")
                    else:
                        rechazadas.append(d); log(f"  ✗ {d['clase']} — falso positivo | {v.get('razon','')}","warn")
                    logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                dets=validadas
                pipeline_det.markdown(f"**{len(dets)} confirmados** — {len(rechazadas)} descartados")
                logs_ph.markdown(render_logs(),unsafe_allow_html=True)

            pipeline_txt.markdown("🚗 **Identificando vehículo...**"); pipeline_bar.progress(88)
            log("Gemini Vision → identificando marca y modelo","info"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
            modelo_auto=identificar_modelo(imagen,client)
            log(f"Modelo: {modelo_auto}","ok"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)

            costos=None
            if usar_costos and estimator:
                pipeline_txt.markdown("💶 **Calculando costos...**"); pipeline_bar.progress(95)
                log("CostEstimator → calculando","info"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                costos=estimar_costos(dets,imagen,modelo_auto,estimator)
                if costos: log(f"Costos → {costos['total_min']}–{costos['total_max']} {costos['currency']}","ok")
                logs_ph.markdown(render_logs(),unsafe_allow_html=True)

            if dets:
                pipeline_txt.markdown("🖼️ **Generando visualización...**"); pipeline_bar.progress(98)
                st.session_state["img_renders"] = renderizar_detecciones(imagen, dets, modo="Ambas")
                log("Visualización generada","ok"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
                imagen_ph.image(st.session_state["img_renders"],
                                caption="Detecciones — YOLO26 + SAM 3", use_container_width=True)

            pipeline_bar.progress(100); pipeline_txt.markdown("✅ **Análisis completado**"); pipeline_det.empty()
            log("Pipeline completado","ok"); logs_ph.markdown(render_logs(),unsafe_allow_html=True)
            st.session_state["img_dets"]=dets; st.session_state["img_modelo"]=modelo_auto
            st.session_state["img_costos"]=costos; st.session_state["img_rechazadas"]=rechazadas

        except Exception as e:
            pipeline_txt.markdown("❌ **Error**")
            log(f"Error crítico: {traceback.format_exc()}","error")
            logs_ph.markdown(render_logs(),unsafe_allow_html=True)
            st.error(f"Error: {e}"); st.stop()

    if st.session_state["img_dets"] is not None:
        dets=st.session_state["img_dets"]; modelo_auto=st.session_state["img_modelo"]
        costos=st.session_state["img_costos"]; rechazadas=st.session_state["img_rechazadas"]
        st.markdown("---")
        c1,c2,c3,c4=st.columns(4)
        c1.metric("Vehículo",modelo_auto.split()[0] if modelo_auto else "—"," ".join(modelo_auto.split()[1:]) if modelo_auto else "")
        c2.metric("Daños detectados",len(dets))
        c3.metric("Falsos positivos",len(rechazadas))
        if costos: c4.metric("Costo estimado",f"{costos['total_min']}–{costos['total_max']} {costos['currency']}")
        if dets: st.markdown("### Desglose de daños"); mostrar_desglose(dets, imagen, costos)
        if costos:
            st.markdown("---")
            c_tot,c_hor=st.columns(2)
            c_tot.metric("Total estimado",f"{costos['total_min']}–{costos['total_max']} {costos['currency']}")
            c_hor.metric("Mano de obra",f"{costos['labor_hours_min']}–{costos['labor_hours_max']}h")
        st.markdown("---")
        with st.expander("📋 Logs técnicos completos",expanded=False):
            st.markdown(render_logs(),unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════════════════════
# MODO VIDEO
# ══════════════════════════════════════════════════════════════════════════════
else:
    vid_bytes = st.session_state.get("vid_file_bytes")
    if vid_bytes is None:
        st.info("Sube un video en el panel izquierdo para comenzar.")
        st.stop()

    # mostramos el video original
    vid_tmp = st.session_state.get("vid_tmp_path")
    if vid_tmp and os.path.exists(vid_tmp):
        st.video(vid_tmp)

    if analizar_vid:
        try:
            import cv2

            tmp_path = st.session_state.get("vid_tmp_path", "/content/uploaded_video.mp4")

            cap=cv2.VideoCapture(tmp_path)
            fps_vid=cap.get(cv2.CAP_PROP_FPS) or 25
            total_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            w_vid=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h_vid=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            duracion=total_frames/fps_vid
            st.info(f"Video: {duracion:.1f}s · {w_vid}×{h_vid} · {total_frames} frames totales")

            yolo_model=cargar_yolo(); sam_model=cargar_sam()
            client_vid=cargar_gemini(); estimator_vid=cargar_estimator()

            st.session_state["logs"] = []
            col_vprog, col_vlogs = st.columns([3,2])
            with col_vprog:
                vid_bar=st.progress(0); vid_txt=st.empty()
            with col_vlogs:
                st.markdown("**Logs técnicos**")
                vlogs_ph=st.empty(); vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)

            # ── paso 1: YOLO26 en todos los frames ───────────────────────────
            vid_txt.markdown("🎥 **Paso 1/4 — YOLO26 detectando en todos los frames...**")
            # guardamos dets por frame_idx para usarlas al generar el video
            dets_por_frame={}   # frame_idx → list of dets
            frames_con_danos=[] # solo frames con detecciones (para galería y pipeline)
            frame_idx=0

            while True:
                ret,frame_bgr=cap.read()
                if not ret: break
                frame_rgb=cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                ts=frame_idx/fps_vid
                dets_frame=detectar_yolo(frame_rgb, conf_vid, yolo_model)
                if dets_frame:
                    dets_por_frame[frame_idx]={"dets":dets_frame,"frame":frame_rgb,"ts":ts}
                frame_idx+=1
                progreso=min(int(frame_idx/max(total_frames,1)*35),34)
                vid_bar.progress(progreso)
                if frame_idx % 15 == 0:
                    vid_txt.markdown(f"🎥 **Paso 1/4 — YOLO26** frame {frame_idx}/{total_frames} | {len(dets_por_frame)} con daños")
            cap.release()

            log(f"YOLO26 → {len(dets_por_frame)}/{total_frames} frames con detecciones","ok")
            vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)
            vid_txt.markdown(f"🎥 **Paso 1/4 completado** — {len(dets_por_frame)}/{total_frames} frames con detecciones")

            # ── paso 2: SAM 3 solo en frames con detecciones ─────────────────
            vid_txt.markdown("✂️ **Paso 2/4 — SAM 3 segmentando frames con daños...**")
            frames_procesados_sam=0
            for fi, data in dets_por_frame.items():
                data["dets"]=segmentar(data["frame"], data["dets"], sam_model)
                frames_con_danos.append({"timestamp":data["ts"],"frame":data["frame"],"dets":data["dets"]})
                frames_procesados_sam+=1
                progreso=35+min(int(frames_procesados_sam/max(len(dets_por_frame),1)*20),19)
                vid_bar.progress(progreso)
                vid_txt.markdown(f"✂️ **Paso 2/4 — SAM 3** {frames_procesados_sam}/{len(dets_por_frame)} frames segmentados")
                if frames_procesados_sam % 10 == 0:
                    log(f"SAM3 → {frames_procesados_sam}/{len(dets_por_frame)} frames segmentados","info")
                    vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)

            log(f"SAM3 completado — {len(frames_con_danos)} frames con máscaras","ok")
            vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)

            # ── paso 3: agregar daños únicos + juez VLM ──────────────────────
            vid_txt.markdown("🧩 **Paso 3/4 — Agregando daños únicos y verificando...**"); vid_bar.progress(56)
            todas_dets_vid=[]
            for fr in frames_con_danos:
                for d in fr["dets"]: todas_dets_vid.append({**d,"frame_ref":fr["frame"]})
            # dedup por clase — en video el mismo daño aparece con bboxes distintos
            # por movimiento de cámara. Guardamos el de mayor confianza por clase.
            mejor_por_clase = {}
            for det in todas_dets_vid:
                clase = det["clase"]
                if clase not in mejor_por_clase or float(det["confianza"]) > float(mejor_por_clase[clase]["confianza"]):
                    mejor_por_clase[clase] = det
            dets_unicas = list(mejor_por_clase.values())
            log(f"Agregación → {len(todas_dets_vid)} detecciones → {len(dets_unicas)} daños únicos","ok")
            vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)
            vid_txt.markdown(f"🧩 {len(todas_dets_vid)} detecciones → {len(dets_unicas)} daños únicos — aplicando juez VLM...")
            vid_bar.progress(60)

            # ordenamos por confianza desc y limitamos a 6 para no agotar quota Gemini
            dets_unicas = sorted(dets_unicas, key=lambda d: float(d["confianza"]), reverse=True)[:6]
            validadas_vid=[]; rechazadas_vid=[]
            for i,d in enumerate(dets_unicas):
                v=juez_vlm(d["frame_ref"],d,client_vid); d["juez"]=v
                if v.get("es_dano") or v.get("confianza",1)<0.6: validadas_vid.append(d)
                else: rechazadas_vid.append(d)
                vid_bar.progress(60+int((i+1)/max(len(dets_unicas),1)*15))
                vid_txt.markdown(f"🧠 **Juez VLM** {i+1}/{len(dets_unicas)} | ✓{len(validadas_vid)} ✗{len(rechazadas_vid)}")
                log(f"  {'✓' if v.get('es_dano') else '✗'} {d['clase']} (conf_juez={v.get('confianza',0):.2f})","ok" if v.get('es_dano') else "warn")
                vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)

            # ── paso 4: modelo, costos y video output frame a frame ───────────
            vid_txt.markdown("🚗 **Paso 4/4 — Identificando vehículo y calculando costos...**"); vid_bar.progress(76)
            frame_ref_mid=frames_con_danos[len(frames_con_danos)//2]["frame"] if frames_con_danos else None
            modelo_vid="Modelo no identificado"
            if frame_ref_mid is not None: modelo_vid=identificar_modelo(frame_ref_mid,client_vid)
            log(f"Modelo identificado: {modelo_vid}","ok")
            vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)
            vid_bar.progress(80)
            costos_vid=None
            if validadas_vid and estimator_vid:
                costos_vid=estimar_costos(validadas_vid,frame_ref_mid,modelo_vid,estimator_vid)
                if costos_vid: log(f"Costos → {costos_vid['total_min']}–{costos_vid['total_max']} {costos_vid['currency']}","ok")
                vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)

            # video output — pintamos TODOS los frames con las dets de ese frame
            vid_txt.markdown("🎬 **Paso 4/4 — Generando video frame a frame...**"); vid_bar.progress(83)
            output_path="/content/video_output.mp4"
            fourcc=cv2.VideoWriter_fourcc(*'mp4v')
            out=cv2.VideoWriter(output_path,fourcc,fps_vid,(w_vid,h_vid))
            cap2=cv2.VideoCapture(tmp_path)
            frame_idx2=0

            while True:
                ret,frame_bgr=cap2.read()
                if not ret: break
                # si este frame tiene detecciones las pintamos
                if frame_idx2 in dets_por_frame:
                    for det in dets_por_frame[frame_idx2]["dets"]:
                        x1,y1,x2,y2=det["caja"].astype(int)
                        color_rgb=COLORES_CLASE.get(det["clase"],(140,140,140))
                        color_bgr=(color_rgb[2],color_rgb[1],color_rgb[0])
                        # máscara SAM primero (debajo del bbox)
                        if det.get("mascara") is not None:
                            mask=det["mascara"]
                            if mask.shape[:2]==(h_vid,w_vid):
                                overlay=frame_bgr.copy()
                                overlay[mask>0]=np.clip(
                                    overlay[mask>0]*0.5+np.array(color_bgr)*0.5,0,255
                                ).astype(np.uint8)
                                cv2.addWeighted(overlay,0.45,frame_bgr,0.55,0,frame_bgr)
                        # bbox y label encima
                        cv2.rectangle(frame_bgr,(x1,y1),(x2,y2),color_bgr,2)
                        label=f"{det['clase']} {det['confianza']:.2f}"
                        (tw,th),_=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.55,2)
                        cv2.rectangle(frame_bgr,(x1,max(y1-th-8,0)),(x1+tw+4,y1),(0,0,0),-1)
                        cv2.putText(frame_bgr,label,(x1+2,max(y1-4,12)),
                                   cv2.FONT_HERSHEY_SIMPLEX,0.55,color_bgr,2)
                out.write(frame_bgr)
                frame_idx2+=1
                if frame_idx2 % 30 == 0:
                    progreso=83+min(int(frame_idx2/max(total_frames,1)*15),14)
                    vid_bar.progress(progreso)
                    vid_txt.markdown(f"🎬 **Generando video...** frame {frame_idx2}/{total_frames}")

            cap2.release(); out.release()

            log("Pipeline de video completado","ok")
            vlogs_ph.markdown(render_logs(),unsafe_allow_html=True)
            vid_bar.progress(100)
            vid_txt.markdown(f"✅ **Análisis completado** — {len(validadas_vid)} daños confirmados en el vehículo")
            st.session_state["vid_resultados"]=frames_con_danos
            st.session_state["vid_resumen"]={"dets_unicas":validadas_vid,"rechazadas":rechazadas_vid,
                                              "modelo":modelo_vid,"costos":costos_vid}
            st.session_state["vid_output_path"]=output_path

        except Exception as e:
            st.error(f"Error procesando video: {e}")
            st.caption(traceback.format_exc())

    if st.session_state.get("vid_resultados") is not None:
        frames_con_danos=st.session_state["vid_resultados"]
        resumen=st.session_state.get("vid_resumen",{})
        dets_unicas=resumen.get("dets_unicas",[])
        rechazadas_vid=resumen.get("rechazadas",[])
        modelo_vid=resumen.get("modelo","—")
        costos_vid=resumen.get("costos",None)

        if not frames_con_danos:
            st.warning("No se detectaron daños en ningún frame.")
        else:
            st.markdown("---")
            c1,c2,c3,c4=st.columns(4)
            c1.metric("Vehículo",modelo_vid.split()[0] if modelo_vid else "—"," ".join(modelo_vid.split()[1:]) if modelo_vid else "")
            c2.metric("Daños únicos confirmados",len(dets_unicas))
            c3.metric("Falsos positivos",len(rechazadas_vid))
            if costos_vid: c4.metric("Costo estimado",f"{costos_vid['total_min']}–{costos_vid['total_max']} {costos_vid['currency']}")

            if st.session_state.get("vid_output_path") and os.path.exists(st.session_state["vid_output_path"]):
                st.markdown("### Video con detecciones")
                with open(st.session_state["vid_output_path"],"rb") as vf:
                    st.download_button("⬇️ Descargar video con detecciones",
                                       data=vf, file_name="inspeccion_daños.mp4", mime="video/mp4")

            if dets_unicas:
                st.markdown("### Daños únicos detectados")
                mostrar_desglose(dets_unicas, dets_unicas[0]["frame_ref"], costos_vid)

            if costos_vid:
                st.markdown("---")
                c_tot,c_hor=st.columns(2)
                c_tot.metric("Total estimado",f"{costos_vid['total_min']}–{costos_vid['total_max']} {costos_vid['currency']}")
                c_hor.metric("Mano de obra",f"{costos_vid['labor_hours_min']}–{costos_vid['labor_hours_max']}h")
            with st.expander("📋 Logs técnicos completos",expanded=False):
                st.markdown(render_logs(),unsafe_allow_html=True)
            st.markdown(f"### Frames con daños ({len(frames_con_danos)})")
            for i in range(0,len(frames_con_danos),3):
                cols=st.columns(3)
                for j,col in enumerate(cols):
                    if i+j>=len(frames_con_danos): break
                    fr=frames_con_danos[i+j]
                    with col:
                        img_r=renderizar_detecciones(fr["frame"],fr["dets"],modo="Ambas")
                        clases_frame=list(set(d["clase"] for d in fr["dets"]))
                        st.image(img_r,caption=f"⏱ {fr['timestamp']:.1f}s — {', '.join(clases_frame)}",
                                 use_container_width=True)

Overwriting app.py


In [9]:
!pip install streamlit pyngrok -q
!pip install ultralytics -q

In [10]:
from google.colab import userdata
import os
os.environ['GEMINI_KEY'] = userdata.get('GeminiKey')
print("Key configurada")

Key configurada


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from pyngrok import ngrok

ngrok.set_auth_token("3Fgt0uadDqTxo2YcCXbQhzYKuVy_84uWJe1c6sCYwPFT4ewUM")

In [13]:
from pyngrok import ngrok

# cerramos todos los túneles abiertos
ngrok.kill()

# relanzamos streamlit limpio
proc.kill()

In [15]:
import subprocess, time
from pyngrok import ngrok

# matar procesos anteriores
ngrok.kill()
try: proc.kill()
except: pass

time.sleep(2)

# lanzar streamlit
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.fileWatcherType", "none"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(4)  # esperamos que arranque

public_url = ngrok.connect(8501)
print(f"App: {public_url}")

App: NgrokTunnel: "https://oval-unstirred-recent.ngrok-free.dev" -> "http://localhost:8501"
